<a href="https://colab.research.google.com/github/knyzk14/maze-visualizer/blob/main/%E8%AA%B2%E9%A1%8C04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **迷路生成・経路探索アルゴリズムの可視化シミュレーター**

## **概要**

本プログラムは、多様なアルゴリズムを用いて迷路を生成し、さらにその迷路を解くまでの探索過程をアニメーションで可視化するインタラクティブなシミュレーターです。

迷路の自動生成には「棒倒し法」、「穴掘り法（再帰的バックトラック）」、および「壁延ばし法」などの代表的なアルゴリズムを実装し、  
それぞれがどのような過程を経て迷路を作り出すのかを視覚的に理解できるようにすることを目指しています。  
また、完成した迷路に対する経路探索（スタートからゴールまでのルート探索）についても、「深さ優先探索（DFS）」、「幅優先探索（BFS）」、「A*（エースター）探索」といった三つのアルゴリズムを選択し、  
挙動の違いを確認することも可能です。

Colaboratory上での対話的な操作をinputを使わず実現するため、  
ipywidgetsモジュールを用いることにしました。  
ユーザーはウィジェット上のスライダーやドロップダウンメニューを通じて、迷路のサイズやアニメーションの描画速度、使用するアルゴリズムを変更しシミュレーションを実行できます。
## **操作方法**

1. 全てのコードセルを上から順に実行してください。

2. 一番下のセルを実行すると、セルの出力領域に操作用のUIが表示されます。

3. 「迷路の幅」「迷路の高さ」「アニメーション速度」をなどを設定できます。

4. **迷路の作成**: 「生成アルゴリズム」のドロップダウンメニューから任意の手法を選択し、「迷路生成」ボタンをクリックします。生成過程がコマ送りで描画されます。


5. **経路の探索**: 迷路が完成した後、「探索アルゴリズム」のドロップダウンメニューから任意の手法を選択し、「探索開始」ボタンをクリックします。探索済みのルートと最終的な最短ルートが描画されます。

## **制作者のコメント**

### 工夫について
本プログラムの最大の工夫は、迷路の「生成・探索ロジック」と「UI描画」をジェネレータ (`yield`) を用いて完全に分離している部分です。各アルゴリズム関数内に `print` や `time.sleep` を直接書かず、毎ステップの配列のみを返す設計にしたことで、後半のBFSやA*といった複雑な探索アルゴリズムの拡張もスムーズに行えました。

### 苦労した部分
「壁延ばし法」に苦戦しました。ランダムに壁を延ばす過程で「自分が直前に置いた壁」を巻き込んで自己衝突してしまい、無限ループに陥るバグ等に悩まされました。
その時の挙動がヘビのようで見ている分には面白かったのですが...  
最終的に、進行可能方向のリストアップ条件を厳密に管理し、袋小路に入った時だけスタックを初期化してロールバックする処理を実装することで突破できました。

またA*探索アルゴリズムが難解で、`heapq`モジュールも用いたことが無かったため理解と実装に苦戦しました。

### 感想
今回初めて `ipywidgets` を使ったイベント駆動のUI構築に挑戦しました。html,cssを用いた経験はあるものの、python上でhtml、それに類するものを記述したのは初めてで楽しかったです。

## **作成時間**

作成時間：**約12時間**

In [ ]:
%reset -f
!pip install ipykernel ipywidgets IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 31.1 MB/s eta 0:00:00


In [ ]:
import ipywidgets as widgets
from IPython.display import display,HTML,clear_output
from copy import deepcopy as dc
import time,random
import heapq

# ===============================================

display(HTML("""
<style>
.maze-output-style {
    font-family: monospace;
    line-height: 1.0;
    letter-spacing: 0px;
    margin: 0;
    padding: 10px;
    color: #a9b7c6;
    font-size: large;
}
</style>
"""))

# ===============================================

# 入力・操作コンポーネントの作成

# ===============================================

# スライダー類
width_slider = widgets.IntSlider(value=21, min=9, max=61, step=2, description='幅:')
height_slider = widgets.IntSlider(value=21, min=9, max=61, step=2, description='高さ:')
speed_slider = widgets.FloatSlider(value=0.05, min=0.01, max=0.5, step=0.01, description='描画間隔(秒):')

# ドロップダウン
gen_algo_dropdown = widgets.Dropdown(
    options=['棒倒し法', '穴掘り法', '壁延ばし法'],
    value='棒倒し法',
    description='生成手法:',
    layout=widgets.Layout(width='200px')
)
solve_algo_dropdown = widgets.Dropdown(
    options=['深さ優先探索(DFS)', '幅優先探索(BFS)', 'A*探索'],
    value='深さ優先探索(DFS)',
    description='探索手法:',
    layout=widgets.Layout(width='230px')
)

btn_generate = widgets.Button(description='迷路生成', button_style='info', icon='hammer')
btn_solve = widgets.Button(description='探索開始', button_style='success', icon='search')

output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', height='auto', padding='10px'))

output_area.add_class('maze-output-style')

settings_box = widgets.HBox([width_slider, height_slider, speed_slider])
generate_box = widgets.HBox([gen_algo_dropdown, btn_generate], layout=widgets.Layout(margin='10px 0px'))
solve_box = widgets.HBox([solve_algo_dropdown, btn_solve], layout=widgets.Layout(margin='10px 0px'))

control_panel = widgets.VBox([
    settings_box,
    widgets.HBox([generate_box, solve_box], layout=widgets.Layout(justify_content='center'))
])

# ===============================================

# ジェネレーターの作成

# ===============================================

def stick_falling_gen(w,h):
    # 初期化
    maze = [["  " for i in range(w)] for i in range(h)]

    yield dc(maze)

    # 外周を全て壁にする

    for i in range(len(maze)):
        if i == 0 or i == len(maze) - 1:
            maze[i] = ["■" for i in range(w)]
        else:
            maze[i][0] = "■"
            maze[i][-1] = "■"

        yield dc(maze)

    for i in range(len(maze)):
        if i % 2 != 0 or i == 0 or i == len(maze) - 1:
            continue
        for j in range(len(maze[i])):
            if j % 2 != 0:
                continue
            else:
                maze[i][j] = "■"
                yield dc(maze)

    for i in range(len(maze)):
        if i % 2 != 0 or i == 0 or i == len(maze) - 1:
            continue
        for j in range(len(maze[i])):
            if j % 2 != 0 or j == 0 or j == len(maze[i]) - 1:
                continue
            else:
                if i == 2:
                    directions = [(-1,0),(1,0),(0,-1),(0,1)]
                else:
                    directions = [(1,0),(0,-1),(0,1)]

                while True:
                    direction = random.choice(directions)
                    target_i = i + direction[0]
                    target_j = j + direction[1]

                    if maze[target_i][target_j] != "■":
                        break

                maze[i+direction[0]][j+direction[1]] = "■"
                yield dc(maze)

    yield dc(maze)

def digging_gen(w,h):
    maze = [["■" for i in range(w)] for i in range(h)]
    yield dc(maze)

    stack = []

    maze[h-2][w-2] = "☆"
    stack.append((h-2,w-2))
    yield dc(maze)

    while True:
        current = stack[-1]
        directions = [(-2, 0), (2, 0), (0, -2), (0, 2)]
        diggable_directions = []

        for dy, dx in directions:
            # 2マス先の座標を計算
            target_y = current[0] + dy
            target_x = current[1] + dx

            if 0 < target_y < h - 1 and 0 < target_x < w - 1:
                if maze[target_y][target_x] == "■":
                    # 条件を満たせば、候補に追加
                    diggable_directions.append((dy, dx))

        if diggable_directions:
            direction = random.choice(diggable_directions)
            dy = direction[0]
            dx = direction[1]
            ddy = dy+1 if dy < 0 else dy-1 if dy > 0 else 0
            ddx = dx+1 if dx < 0 else dx-1 if dx > 0 else 0
            maze[current[0] + dy][current[1] + dx] = "★"
            maze[current[0] + ddy][current[1] + ddx] = "  "
            stack.append((current[0] + dy,current[1] + dx))
            yield dc(maze)
        else:
            maze[current[0]][current[1]] = "  "
            stack.pop()
            yield dc(maze)
            if not len(stack):
                break
            else:
                continue

def wall_extension_gen(w,h):
    # 初期化
    maze = [["  " for i in range(w)] for i in range(h)]

    yield dc(maze)

    # 外周を全て壁にする

    for i in range(len(maze)):
        if i == 0 or i == len(maze) - 1:
            maze[i] = ["■" for i in range(w)]
        else:
            maze[i][0] = "■"
            maze[i][-1] = "■"

        yield dc(maze)

    stack = []

    while True:
        canditates = [(y, x) for y in range(2, h-1, 2) for x in range(2, w-1, 2) if maze[y][x] != "■"]
        if not canditates:
            break
        else:
            starting_point = random.choice(canditates)
            maze[starting_point[0]][starting_point[1]] = "＋"
            stack.append(starting_point)
            yield dc(maze)
            while True:
                current = stack[-1]
                directions = [(-2, 0), (2, 0), (0, -2), (0, 2)]
                diggable_directions = []

                for dy, dx in directions:
                    target_y = current[0] + dy
                    target_x = current[1] + dx

                    if 0 <= target_y <= h - 1 and 0 <= target_x <= w - 1:
                        if maze[target_y][target_x] != "＋":
                            diggable_directions.append((dy, dx))
                if not diggable_directions:
                    maze[current[0]][current[1]] = "□"
                    yield dc(maze)
                    for y, x in stack:
                        maze[y][x] = "  "
                        yield dc(maze)
                    stack = [starting_point]
                    maze[starting_point[0]][starting_point[1]] = "＋"
                    continue

                direction = random.choice(diggable_directions)
                dy = direction[0]
                dx = direction[1]
                ddy = dy+1 if dy < 0 else dy-1 if dy > 0 else 0
                ddx = dx+1 if dx < 0 else dx-1 if dx > 0 else 0
                maze[current[0] + ddy][current[1] + ddx] = "＋"
                stack.append((current[0] + ddy,current[1] + ddx))
                yield dc(maze)

                target_y = current[0] + dy
                target_x = current[1] + dx

                if maze[target_y][target_x] == "  ":
                    maze[target_y][target_x] = "＋"
                    stack.append((target_y, target_x))
                    continue
                elif maze[target_y][target_x] == "■":
                    for y, x in stack:
                        maze[y][x] = "■"
                        yield dc(maze)
                    stack = []
                    break

            yield dc(maze)

def solve_dfs_gen(maze, w, h):
    start_y, start_x = 1, 1
    goal_y, goal_x = h - 2, w - 2

    stack = [(start_y, start_x)]

    maze[start_y][start_x] = "＠"
    yield dc(maze)

    while stack:
        current = stack[-1]

        if current[0] == goal_y and current[1] == goal_x:
            break

        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        moved = False

        for dy, dx in directions:
            target_y = current[0] + dy
            target_x = current[1] + dx

            if maze[target_y][target_x] == "  ":
                maze[target_y][target_x] = "＠"
                stack.append((target_y, target_x))

                moved = True
                yield dc(maze)
                break

        if not moved:
            maze[current[0]][current[1]] = "□"
            stack.pop()

            yield dc(maze)
    yield dc(maze)

def solve_bfs_gen(maze, w, h):
    start_y, start_x = 1, 1
    goal_y, goal_x = h - 2, w - 2

    queue = [(start_y, start_x)]
    came_from = {(start_y, start_x): None}

    maze[start_y][start_x] = "・"
    yield dc(maze)

    found = False

    while queue:
        current = queue.pop(0)

        if current[0] == goal_y and current[1] == goal_x:
            found = True
            break

        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dy, dx in directions:
            target_y = current[0] + dy
            target_x = current[1] + dx

            if maze[target_y][target_x] == "  ":
                maze[target_y][target_x] = "・"
                came_from[(target_y, target_x)] = current
                queue.append((target_y,target_x))
                yield dc(maze)

    if found:
        current = (goal_y, goal_x)
        while current is not None:
            maze[current[0]][current[1]] = "＠"
            current = came_from[current]
            yield dc(maze)
    yield dc(maze)

def solve_astar_gen(maze, w, h):
    start_y, start_x = 1, 1
    goal_y, goal_x = h - 2, w - 2

    # ヒューリスティック関数（現在地からゴールまでのマンハッタン距離）
    def get_h(y, x):
        return abs(goal_y - y) + abs(goal_x - x)

    queue = []
    # (fスコア, gスコア, y座標, x座標)
    heapq.heappush(queue, (get_h(start_y, start_x), 0, start_y, start_x))

    came_from = {(start_y, start_x): None}
    g_scores = {(start_y, start_x): 0}

    maze[start_y][start_x] = "・"
    yield dc(maze)

    found = False

    while queue:
        # fスコアが最も小さいマスを取り出す
        current_f, current_g, current_y, current_x = heapq.heappop(queue)

        if current_y == goal_y and current_x == goal_x:
            found = True
            break

        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        for dy, dx in directions:
            target_y = current_y + dy
            target_x = current_x + dx

            # 次のマスへ進んだ場合のg
            next_g = current_g + 1

            if maze[target_y][target_x] in ["  ", "・"]:
                if (target_y, target_x) not in g_scores or next_g < g_scores[(target_y, target_x)]:
                    g_scores[(target_y, target_x)] = next_g
                    next_f = (next_g + get_h(target_y, target_x))
                    heapq.heappush(queue,(next_f,next_g,target_y,target_x))
                    came_from[(target_y, target_x)] = (current_y, current_x)
                    maze[target_y][target_x] = "・"
                    yield dc(maze)
                pass

    if found:
        current = (goal_y, goal_x)
        while current is not None:
            maze[current[0]][current[1]] = "＠"
            current = came_from[current]
            yield dc(maze)
            pass

    yield dc(maze)
# ===============================================

# UIの描画等

# ===============================================

current_completed_maze = None

def on_generate_clicked(b):
    """迷路生成用の処理"""
    global current_completed_maze
    with output_area:
        clear_output(wait=True)

        w = width_slider.value
        h = height_slider.value
        speed = speed_slider.value
        algo = gen_algo_dropdown.value

        if algo == "棒倒し法":
            generator = stick_falling_gen(w, h)
        elif algo == "穴掘り法":
            generator = digging_gen(w, h)
        else:
            generator = wall_extension_gen(w, h)

        # 迷路を描画
        last_maze = None
        for current_maze in generator:
            clear_output(wait=True)
            for row in current_maze:
                print("".join(row))
            time.sleep(speed)
            last_maze = current_maze

        if last_maze is not None:
            current_completed_maze = dc(last_maze)
            print("\n生成完了！")

def on_solve_clicked(b):
    """探索用の処理"""
    global current_completed_maze
    with output_area:
        if current_completed_maze is None:
            clear_output(wait=True)
            print("先に迷路を生成してください！")
            return

        w = width_slider.value
        h = height_slider.value
        speed = speed_slider.value
        algo = solve_algo_dropdown.value

        maze_to_solve = dc(current_completed_maze)

        if algo == "深さ優先探索(DFS)":
            generator = solve_dfs_gen(maze_to_solve, w, h)
        elif algo == "幅優先探索(BFS)":
            generator = solve_bfs_gen(maze_to_solve, w, h)
        elif algo == "A*探索":
            generator = solve_astar_gen(maze_to_solve, w, h)

        for step_maze in generator:
            clear_output(wait=True)
            for row in step_maze:
                print("".join(row))
            time.sleep(speed)

        print("\n探索完了！")

# ====================================

# 関数の設定

btn_generate.on_click(on_generate_clicked)
btn_solve.on_click(on_solve_clicked)

# run

display(widgets.VBox([control_panel,output_area]))